# 06: 自分のデータで再学習

04の学習開始時と同じSmolVLAを、05で作ったデータで学習します。その後LIBEROで評価します。依存ライブラリ、学習済みモデル、LIBEROのシーンは実行時に取得します。

05でデータを変換し、最後の保存セルで表示される「06のDATASET_ROOT」を控えておきます。

1. Google Driveの「マイドライブ」に `vla_workshop` フォルダを作り、その中に `06_collected_finetuning` をフォルダごと置きます。
2. このノートブックをColabで開き、ランタイムを **T4 GPU** にします。別の場所に置いた場合は、最初のコードセルの `LESSON_DIR` を変更します。
3. 上から順にセルを実行します。

同じフォルダに置くファイル

- `common.py`
- `evaluation.json`
- `libero_env.py`
- `prepare.py`
- `requirements.txt`
- `robot_io.py`
- `smolvla.py`
- `train.json`
- `training.py`

In [ ]:
# @title この回の配布ファイルを準備する
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive, files

USE_DRIVE = True  # @param {type:"boolean"}
LESSON_DIR = "/content/drive/MyDrive/vla_workshop/06_collected_finetuning"  # @param {type:"string"}
REQUIRED_FILES = [
    "common.py",
    "evaluation.json",
    "libero_env.py",
    "prepare.py",
    "requirements.txt",
    "robot_io.py",
    "smolvla.py",
    "train.json",
    "training.py",
]

if USE_DRIVE:
    drive.mount("/content/drive")
    source = Path(LESSON_DIR)
else:
    # ipynb以外の今回の配布ファイルを、まとめて選んでアップロードします。
    os.chdir("/content")
    files.upload()
    source = Path("/content")
missing = [name for name in REQUIRED_FILES if not (source / name).is_file()]
if missing:
    raise FileNotFoundError(f"配布ファイルが不足: {missing}。LESSON_DIRを確認してください。")
PROJECT = Path("/content/lesson_06")
PROJECT.mkdir(exist_ok=True)
for name in REQUIRED_FILES:
    shutil.copy2(source / name, PROJECT / name)
os.chdir(PROJECT)
print("この回の作業場所:", PROJECT)

## ライブラリの準備

次のセルで、この回に必要なライブラリを準備します。初回は数分かかることがあります。

In [ ]:
# @title ライブラリを準備する
LESSON_ENV = {
    **os.environ,
    "MPLBACKEND": "Agg",
    "PYTHONPATH": str(PROJECT),
    "PYTHONNOUSERSITE": "1",
}
command = [sys.executable, str(PROJECT / "prepare.py")]

subprocess.run(command, env=LESSON_ENV, check=True)
PYTHON = PROJECT / ".venv/bin/python"


def run_python(program):
    """セルに書いたPythonを、この回の専用Pythonで実行する。"""
    last_result = None
    with subprocess.Popen(
        [str(PYTHON), "-u", "-c", program],
        cwd=PROJECT,
        env=LESSON_ENV,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            if line.startswith("Results: "):
                last_result = Path(line.removeprefix("Results: ").strip())
        if process.wait():
            raise RuntimeError("実行に失敗しました。直前のエラーを確認してください。")
    return last_result

## モデルを読み込み、行動を予測する処理

ここでは後のセルで使う処理を用意します。モデルを読み込むのは、実習を実行するセルです。

In [ ]:
MODEL_CODE = r"""
import torch
from lerobot.policies.factory import make_pre_post_processors
from lerobot.policies.smolvla import processor_smolvla  # noqa: F401

from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy

from smolvla import checkpoint_inputs, observation_batch


def load_model(settings, image_size=256):
    checkpoint, config, vlm_repo = checkpoint_inputs(settings, image_size)
    policy = SmolVLAPolicy.from_pretrained(checkpoint, config=config, strict=True).eval()
    # 正規化・トークン化・行動の逆正規化はチェックポイント付属のものを使います。
    pre, post = make_pre_post_processors(
        config,
        pretrained_path=checkpoint,
        preprocessor_overrides={
            "device_processor": {"device": config.device},
            "tokenizer_processor": {"tokenizer_name": config.vlm_model_name},
        },
        postprocessor_overrides={"device_processor": {"device": "cpu"}},
    )
    return policy, pre, post, vlm_repo


def predict_action(policy, pre, post, obs, instruction, settings):
    batch = observation_batch(obs, instruction, settings)
    with torch.inference_mode():
        action = post(policy.select_action(pre(batch)))
    return action.squeeze(0).detach().cpu().numpy()

"""

## LIBEROでの評価処理

カメラ画像などの観測から行動を予測し、シミュレータを1ステップ進めます。評価を実行するセルでこの処理を使います。

In [ ]:
EVALUATION_CODE = r"""
import hashlib

import imageio.v2 as imageio
import numpy as np

from common import output_directory, write_json
from libero_env import ASSETS_REVISION, open_scene, scene_cases
from robot_io import validate_action


def evaluate(settings, split):
    policy, pre, post, _ = load_model(settings["policy"])
    folder = output_directory("libero_" + split)
    episodes = []

    for task, initial in scene_cases(settings, split):
        seed = settings["seed"] + task * 10000 + initial
        torch.manual_seed(seed)
        np.random.seed(seed)
        policy.reset()
        pre.reset()
        post.reset()
        env, obs, instruction = open_scene(settings, task, initial, seed)
        try:
            state_hash = hashlib.sha256(np.asarray(env.get_sim_state()).tobytes()).hexdigest()
            video_path = folder / f"task_{task:02d}_init_{initial:03d}.mp4"
            success = False
            with imageio.get_writer(video_path, fps=20) as video:
                for step in range(settings["max_steps"]):
                    video.append_data(obs["agentview_image"][::-1, ::-1])
                    action = predict_action(policy, pre, post, obs, instruction, settings["policy"])
                    obs, _, done, _ = env.step(validate_action(action))
                    success = bool(env.check_success())
                    if success or done:
                        break
                video.append_data(obs["agentview_image"][::-1, ::-1])
            episodes.append({
                "task_id": task,
                "initial_state": initial,
                "seed": seed,
                "initial_state_sha256": state_hash,
                "instruction": instruction,
                "steps": step + 1,
                "success": success,
                "video": video_path.name,
            })
        finally:
            env.close()

    successes = sum(episode["success"] for episode in episodes)
    report = {
        "split": split, "settings": settings, "assets_revision": ASSETS_REVISION,
        "trials": len(episodes), "successes": successes, "success_rate": successes / len(episodes),
        "episodes": episodes,
    }
    write_json(folder / "report.json", report)
    print(f"Results: {folder}", flush=True)
    return folder

"""

## 05で保存したデータの場所を指定する

05の保存セルに表示された「06のDATASET_ROOT」のパスを、下の `DATASET_ROOT` に貼り付けて実行します。初期設定では、変換後の試行0と1を学習、試行2を検証に使います。

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
# 05の保存セルが表示した「06のDATASET_ROOT」を貼り付けます。
DATASET_ROOT = ""  # @param {type:"string"}
if not DATASET_ROOT or not (Path(DATASET_ROOT) / "meta/info.json").is_file():
    raise FileNotFoundError("05で変換したdatasetフォルダを指定してください")
config = json.loads((PROJECT / "train.json").read_text())
config["dataset"]["root"] = DATASET_ROOT
(PROJECT / "active_train.json").write_text(json.dumps(config))
print(json.dumps(config, indent=2, ensure_ascii=False))

## 自分のデータで学習する

In [ ]:
result = run_python(
    MODEL_CODE
    + r"""
import json

from lerobot.utils.collate import lerobot_collate_fn

from common import output_directory, read_json, write_json
from training import load_training_data, save_model

config = read_json("active_train.json")
torch.manual_seed(config["seed"])
policy, pre, post, vlm_repo = load_model(config["policy"])
training, validation = load_training_data(config["dataset"], policy)

# チェックポイントで学習対象になっているパラメータだけを更新します。
parameters = [parameter for parameter in policy.parameters() if parameter.requires_grad]
optimizer = torch.optim.AdamW(parameters, lr=config["learning_rate"])
batch_size = config["batch_size"]
if min(config["steps"], batch_size) < 1:
    raise ValueError("stepsとbatch_sizeは1以上にしてください")

policy.train()
losses = []
for step in range(config["steps"]):
    indices = torch.randint(len(training), (batch_size,)).tolist()
    batch = lerobot_collate_fn([training[index] for index in indices])
    optimizer.zero_grad()
    loss, _ = policy(pre(batch))
    loss.backward()
    torch.nn.utils.clip_grad_norm_(parameters, config["max_grad_norm"], error_if_nonfinite=True)
    optimizer.step()
    row = {"step": step + 1, "train_loss": loss.item()}
    losses.append(row)
    print(row, flush=True)

policy.eval()
held_out = lerobot_collate_fn([validation[i % len(validation)] for i in range(batch_size)])
with torch.inference_mode():
    validation_loss, _ = policy(pre(held_out))
if not torch.isfinite(validation_loss):
    raise ValueError("検証lossが有限ではありません")
losses[-1]["validation_loss"] = validation_loss.item()

folder = output_directory("finetuning")
checkpoint = folder / "pretrained_model"
save_model(policy, pre, post, vlm_repo, checkpoint)
(folder / "losses.jsonl").write_text("".join(json.dumps(row) + "\n" for row in losses))
write_json(folder / "training.json", {"config": config, "checkpoint": str(checkpoint)})
print(f"Results: {folder}", flush=True)
"""
)
training_report = json.loads((result / "training.json").read_text())
print((result / "losses.jsonl").read_text())

## 学習したモデルを04と同じLIBERO条件で評価する

In [ ]:
evaluation = json.loads((PROJECT / "evaluation.json").read_text())
evaluation["policy"] = {
    **training_report["config"]["policy"],
    "path": training_report["checkpoint"],
    "revision": None,
}
(PROJECT / "trained_evaluation.json").write_text(json.dumps(evaluation))
evaluation_result = run_python(
    MODEL_CODE
    + EVALUATION_CODE
    + r"""
from common import read_json

evaluate(read_json("trained_evaluation.json"), split="validation")
"""
)
print((evaluation_result / "report.json").read_text())

## 確認課題

学習・検証に使ったエピソード番号、loss、評価結果を記録してください。

## 結果をDriveに保存する

モデル、loss、評価結果を保存します。07の改善比較でモデルの保存先を使えます。

In [ ]:
# @title 結果を保存する
from datetime import UTC, datetime
from uuid import uuid4

RESULTS_DIR = "/content/drive/MyDrive/vla_workshop/results"  # @param {type:"string"}
drive.mount("/content/drive")
destination = Path(RESULTS_DIR)
destination.mkdir(parents=True, exist_ok=True)
name = (
    "06_collected_finetuning_"
    + datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
    + "_"
    + uuid4().hex[:8]
)
saved = destination / name
shutil.copytree(PROJECT / "outputs", saved)
print("保存先:", saved)
for model in sorted(saved.glob("*/pretrained_model/config.json")):
    print("07のpolicy.path:", model.parent)